In [ ]:
import pandas as pd

df = pd.read_csv("processed_clause_dataset_clean.csv")

df.head()

,contract_title,clause_type,answer_text,question,context,has_clause,clause_length,clean_text,text_length
0,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Document Name,DISTRIBUTOR AGREEMENT,Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,21,distributor agreement,21
1,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Agreement Date,"7th day of September, 1999.",Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,27,7th day of september 1999,25
2,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Effective Date,The term of this Agreement shall be ten (10)...,Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,273,the term of this agreement shall be ten 10 yea...,176
3,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Expiration Date,The term of this Agreement shall be ten (10)...,Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,273,the term of this agreement shall be ten 10 yea...,176
4,LIMEENERGYCO_09_09_1999-EX-10-DISTRIBUTOR AGRE...,Renewal Term,If Distributor comp...,Highlight the parts (if any) of this contract ...,EXHIBIT 10.6\n\n ...,True,338,if distributor complies with all of the terms ...,220


In [ ]:
df["clean_text"] = df["clean_text"].fillna("")

In [ ]:
# Grouping contracts
contracts = (
    df.groupby("contract_title")["clean_text"]
    .apply(lambda x: " ".join(x))
    .reset_index()
)

contracts.head()

,contract_title,clean_text
0,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,co branding and advertising agreement the term...
1,ABILITYINC_06_15_2020-EX-4.25-SERVICES AGREEMENT,each of the foregoing parties is referred to h...
2,ACCELERATEDTECHNOLOGIESHOLDINGCORP_04_24_2003-...,joint venture agreement the parties or joint v...
3,ACCURAYINC_09_01_2010-EX-10.31-DISTRIBUTOR AGR...,multiple linac and multi modality distributor ...
4,ADAMSGOLFINC_03_21_2005-EX-10.17-ENDORSEMENT A...,endorsement agreement the term of this agreeme...


In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
#Generating vecctors for each contract
embeddings = model.encode(
    contracts["clean_text"].tolist(),
    show_progress_bar=True, 
)

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [ ]:
print(embeddings.shape)


(509, 384)


In [ ]:
# 509 contracts
# 384-dimensional vectors

In [ ]:
# Building FAISS Index
import faiss
import numpy as np

embeddings = np.array(
    embeddings
).astype("float32")

In [ ]:
index = faiss.IndexFlatL2(
    embeddings.shape[1]
)

index.add(embeddings)

print(index.ntotal)

509


In [ ]:
# Similarity Search Function
# def find_similar_contracts(
#     query_text,
#     top_k=5
# ):
    
#     query_embedding = model.encode(
#         [query_text]
#     ).astype("float32")
    
#     distances, indices = index.search(
#         query_embedding,
#         top_k
#     )
    
#     return contracts.iloc[
#         indices[0]
#     ][["contract_title"]]

In [ ]:
def find_similar_contracts(
    query_text,
    top_k=5
):

    query_embedding = model.encode(
        [query_text]
    ).astype("float32")

    distances, indices = index.search(
        query_embedding,
        top_k
    )

    results = []

    for idx, dist in zip(indices[0], distances[0]):

        similarity = 1 / (1 + dist)

        results.append({
            "contract_title":
                contracts.iloc[idx]["contract_title"],

            "similarity_percent":
                round(similarity * 100, 2)
        })

    return pd.DataFrame(results)

In [ ]:
sample_contract = """
DISTRIBUTION AGREEMENT

This Agreement is entered into between ABC Corporation and XYZ Limited.

The Distributor shall have the exclusive right to market and sell the Company's products.
"""

find_similar_contracts(sample_contract)

,contract_title,similarity_percent
0,"NETGEAR,INC_04_21_2003-EX-10.16-DISTRIBUTOR AG...",54.389999
1,"XLITECHNOLOGIES,INC_12_02_2015-EX-10.02-STRATE...",54.310001
2,LUCIDINC_04_15_2011-EX-10.9-DISTRIBUTOR AGREEMENT,53.320000
3,AudibleInc_20001113_10-Q_EX-10.32_2599586_EX-1...,53.070000
4,XYBERNAUTCORP_07_12_2002-EX-4-SPONSORSHIP AGRE...,52.639999


In [ ]:
sample_contract = contracts.iloc[0]["clean_text"]

find_similar_contracts(sample_contract)

,contract_title,similarity_percent
0,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,100.000000
1,EmbarkComInc_19991008_S-1A_EX-10.10_6487661_EX...,65.570000
2,StampscomInc_20001114_10-Q_EX-10.47_2631630_EX...,63.070000
3,ImpresseCorp_20000322_S-1A_EX-10.11_5199234_EX...,62.990002
4,StaarSurgicalCompany_20180801_10-Q_EX-10.37_11...,62.220001


In [ ]:
sample_contract = contracts.iloc[0][
    "clean_text"
]

find_similar_contracts(
    sample_contract
)

,contract_title,similarity_percent
0,2ThemartComInc_19990826_10-12G_EX-10.10_670028...,100.000000
1,EmbarkComInc_19991008_S-1A_EX-10.10_6487661_EX...,65.570000
2,StampscomInc_20001114_10-Q_EX-10.47_2631630_EX...,63.070000
3,ImpresseCorp_20000322_S-1A_EX-10.11_5199234_EX...,62.990002
4,StaarSurgicalCompany_20180801_10-Q_EX-10.37_11...,62.220001


In [ ]:
# Save Index
faiss.write_index(
    index,
    "contracts_faiss.index"
)

In [ ]:
# Saving the embeddings
contracts.to_pickle(
    "contracts_metadata.pkl"
)